In [ ]:
%pip install semantic-link-labs

### Import packages and functions

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model
from sempy_labs.directlake._generate_shared_expression import generate_shared_expression
from sempy_labs.mirrored_azure_databricks_catalog._list_objects import list_databricks_columns
from sempy_labs.mirrored_azure_databricks_catalog._items import (
    get_mirrored_azure_databricks_catalog, 
    list_mirrored_azure_databricks_catalog_shortcuts,
    create_mirrored_azure_databricks_catalog,
    list_mirrored_azure_databricks_catalogs,
    )
from sempy_labs.connection._databricks import create_azure_databricks_workspace_connection
from sempy_labs.mirrored_azure_databricks_catalog._semantic_model import (
    create_expression_name,
    check_tables_format,
    convert_column_data_type,
    infer_relationships,
)
from sempy_labs.connection._items import list_connections
from sempy_labs.mirrored_azure_databricks_catalog._refresh_catalog_metadata import refresh_catalog_metadata
from sempy_labs._sql import ConnectMirroredAzureDatabricksCatalog

### Set parameters

In [ ]:
# From Azure Databricks
databricks_token = ''
databricks_workspace = ''

# Tables from Azure Databricks to be added to the semantic model (must be in catalog.schema.table format)
tables = [
    "mkcat.dbo.geography",
    "mkcat.dbo.sales",
    "mkcatdim.silver.customer",
]

check_tables_format(tables)

# For Fabric
databricks_connection_name = 'my_databricks_connection'
mirrored_azure_databricks_catalog_workspace = None

# Set for creating semantic model
semantic_model_name = 'mirrortesting2' # Name of the semantic model
semantic_model_workspace = None # Name or ID of the workspace
infer_relationships = True

### Create a connection to the Azure Databricks workspace if one does not exist

In [ ]:
databricks_workspace = databricks_workspace.rstrip('/')
df = list_connections()
df_filt = df[(df['Connection Type'] == 'AzureDatabricksWorkspace') & (df['Connection Path'].str.rstrip('/') == databricks_workspace)]
if df_filt.empty:
    connection_id = create_azure_databricks_workspace_connection(
        name=databricks_connection_name,
        url=databricks_workspace,
        databricks_token=databricks_token,
        privacy_level=None,
    )
else:
    connection_id = df_filt['Connection Id'].iloc[0]

### Create Mirrors for each catalog (if the mirrors do not exist)

In [ ]:
catalogs = list(set([catalog.split('.')[0] for catalog in tables]))

df = list_mirrored_azure_databricks_catalogs(workspace=mirrored_azure_databricks_catalog_workspace)

mirror_ids = {}
for catalog in catalogs:
    df_filt = df[(df['Catalog Name'] == catalog) & (df['Databricks Workspace Connection Id'] == connection_id)]
    if df_filt.empty:
        id = create_mirrored_azure_databricks_catalog(
            name=catalog,
            catalog_name=catalog,
            databricks_workspace_connection_id=connection_id,
            mirroring_mode="Full",
            workspace=mirrored_azure_databricks_catalog_workspace,
        )
        mirror_ids[catalog] = id
    else:
        mirror_ids[catalog] = df_filt['Mirrored Azure Databricks Catalog Id'].iloc[0]

### Refresh Mirrored Azure Database Catalogs

In [ ]:
for catalog, catalog_id in mirror_ids.items():
    refresh_catalog_metadata(
        mirrored_azure_databricks_catalog=catalog_id,
        workspace=mirrored_azure_databricks_catalog_workspace,
    )

### Create the initial (blank) semantic model

In [ ]:
labs.create_blank_semantic_model(dataset=semantic_model_name, workspace=semantic_model_workspace)

### Create the tables and columns in the semantic model

In [ ]:
with connect_semantic_model(dataset=semantic_model_name, workspace=semantic_model_workspace, readonly=False) as tom:

    column_list = []
    for t in tables:
        parts = t.split('.')        
        catalog_name, schema_name, table_name = parts
        catalog_id = mirror_ids[catalog_name]

        # Generate the expression for the Mirrored Azure Databricks Catalog
        expr = generate_shared_expression(
            item=catalog_id,
            item_type='MirroredAzureDatabricksCatalog',
            workspace=mirrored_azure_databricks_catalog_workspace,
            use_sql_endpoint=False
        )

        # Check if the expression already exists in the model
        expression_names = []
        found = False
        expression_name = None
        for e in tom.model.Expressions:
            expression_names.append(e.Name)
            if e.Expression == expr:
                found = True
                expression_name = e.Name

        # Add the expression if it does not exist
        if not found:
            expression_name = create_expression_name(
                expression_names=expression_names
            )
            tom.add_expression(name=expression_name, expression=expr)

        # Determine the columns in the table
        df = list_databricks_columns(databricks_workspace=databricks_workspace, unity_catalog=catalog_name, schema=schema_name, databricks_token=databricks_token)
        df_filt = df[(df['Table Name'] == table_name) & (df['Schema Name'] == schema_name)]

        # Only add table if it is found and has columns
        if df_filt.empty:
            print(f"No columns found for table {t}. Skipping.")
            continue
        if table_name not in [t.Name for t in tom.model.Tables]:
            tom.add_table(name=table_name)
            tom.add_entity_partition(table_name=table_name, entity_name=table_name, expression=expression_name, schema_name=schema_name)

            # Add the columns to the table
            for _, row in df_filt.iterrows():
                column_name = row['Column Name']
                data_type = row['Data Type']
                data_type_converted = convert_column_data_type(data_type)
                column_list.append({
                    "sourceCatalog": catalog_id,
                    "sourceSchema": schema_name,
                    "tableName": table_name,
                    "columnName": column_name,
                    "dataType": data_type_converted,
                })
                if column_name not in [c.Name for c in tom.all_columns() if c.Name == column_name and c.Parent.Name == table_name]:
                    tom.add_data_column(table_name=table_name, column_name=column_name, source_column=column_name, data_type=data_type_converted)

    # Infer relationships
    if infer_relationships:
        relationships = infer_relationships(column_list=column_list, workspace=mirrored_azure_databricks_catalog_workspace)
        for r in relationships:
            tom.add_relationship(
                from_table=r.get('fromTable'),
                from_column=r.get('fromColumn'),
                to_table=r.get('toTable'),
                to_column=r.get('toColumn'),
            )

### Refresh the semantic model

In [ ]:
labs.refresh_semantic_model(dataset=semantic_model_name, workspace=semantic_model_workspace)